# Batch ROI Fitting & Stitching
Per-tile fitting with pooled machine IRF, assembled into full-ROI maps.

**Workflow:**
1. Discover XLIF metadata files
2. For each ROI: fit all tiles independently (pooled machine IRF)
3. Assemble pixel maps with intensity-weighted overlap averaging
4. Derive global lifetime summary
5. Export intensity image, tau-coloured image (0–5 ns), and fit CSV

**Settings** — edit the config cell below, then `Run All`.

| Parameter | Value |
|-----------|-------|
| Exponentials | 3 |
| τ range | 0.145 – 45 ns |
| IRF | Machine IRF (pooled peak-aligned) |
| Tile overlap | Intensity-weighted averaging |
| Output | Intensity TIFF, τ-coloured PNG, global fit CSV |

In [1]:
from pathlib import Path

# ── Edit these ────────────────────────────────────────────────────────────────
XLIF_FOLDER      = Path("/Volumes/Lexar/20260316 Falcon data/xlifs")
PTU_FOLDER       = Path("/Volumes/Lexar/20260316 Falcon data/PTU.sptw")
OUTPUT_BASE      = Path("/Volumes/Lexar/20260316 Falcon data/Results")

# Fitting parameters
N_EXP            = 3
TAU_MIN_NS       = 0.145
TAU_MAX_NS       = 45.0

# Lifetime colour image
TAU_COLOUR_MIN   = 0.0
TAU_COLOUR_MAX   = 5.0

# Machine IRF — leave None to use FLIMKit default
MACHINE_IRF_PATH = None

# DE optimiser
DE_POPSIZE       = 15
DE_MAXITER       = 1000
WORKERS          = -1

# ── Masking & thresholds ───────────────────────────────────────────────────────
#
#  INTENSITY_THRESHOLD : Background exclusion mask (photons/pixel).
#                        Pixels below this are zeroed BEFORE fitting in both
#                        passes — affects what goes into the pooled summed
#                        decay (consensus τ) and which pixels get per-pixel fits.
#                        Set to None to use all pixels (no background mask).
#                        Typical value: 5–20 photons depending on signal level.
#
#  MIN_PHOTONS         : Per-pixel NNLS quality gate (photons/pixel).
#                        Pixels below this are skipped inside fit_per_pixel
#                        and receive NaN in the lifetime maps.
#                        Independent of INTENSITY_THRESHOLD.
#                        None = use MIN_PHOTONS_PERPIX from configs.
#
INTENSITY_THRESHOLD = 3   # e.g. 10  — background mask (both passes)
MIN_PHOTONS         = None   # e.g. 10  — per-pixel NNLS gate (Pass 2 only)

ROTATE_TILES     = True

# TIFF display range — display scaling only, does NOT affect fitting
INTENSITY_DISPLAY_MAX = None   # photons (None = auto 99th percentile)
TAU_DISPLAY_MAX       = TAU_COLOUR_MAX

print(f"XLIF folder : {XLIF_FOLDER}")
print(f"PTU folder  : {PTU_FOLDER}")
print(f"Output base : {OUTPUT_BASE}")


XLIF folder : /Volumes/Lexar/20260316 Falcon data/xlifs
PTU folder  : /Volumes/Lexar/20260316 Falcon data/PTU.sptw
Output base : /Volumes/Lexar/20260316 Falcon data/Results


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from flimkit.PTU.reader   import PTUFile
from flimkit.FLIM.fitters import fit_summed
from flimkit.configs      import (
    MACHINE_IRF_DEFAULT_PATH,
    MACHINE_IRF_FIT_BG, MACHINE_IRF_FIT_SIGMA, MACHINE_IRF_FIT_TAIL,
    MIN_PHOTONS_PERPIX,
)

# Resolve thresholds
_MIN_PHOTONS   = MIN_PHOTONS         if MIN_PHOTONS         is not None else MIN_PHOTONS_PERPIX
_INT_THRESHOLD = INTENSITY_THRESHOLD  # None = no background mask

# Load machine IRF
_irf_path = Path(MACHINE_IRF_PATH) if MACHINE_IRF_PATH else Path(MACHINE_IRF_DEFAULT_PATH)
if not _irf_path.exists():
    raise FileNotFoundError(f"Machine IRF not found: {_irf_path}")

_machine_irf_raw = np.load(str(_irf_path)).ravel().astype(float)
_machine_irf_raw = np.maximum(_machine_irf_raw, 0.0)
_machine_irf_raw /= _machine_irf_raw.sum()
PI_MACHINE = int(np.argmax(_machine_irf_raw))

print(f"Machine IRF       : {_irf_path.name}  |  {len(_machine_irf_raw)} bins  |  peak bin {PI_MACHINE}")
print(f"FIT_BG={MACHINE_IRF_FIT_BG}  FIT_SIGMA={MACHINE_IRF_FIT_SIGMA}  HAS_TAIL={MACHINE_IRF_FIT_TAIL}")
print(f"n_exp={N_EXP}  tau=[{TAU_MIN_NS}, {TAU_MAX_NS}] ns")
print()
print(f"INTENSITY_THRESHOLD : {_INT_THRESHOLD}  "
      f"{'(background mask — applied to both passes)' if _INT_THRESHOLD else '(disabled — all pixels used)'}")
print(f"MIN_PHOTONS (NNLS)  : {_MIN_PHOTONS}  "
      f"(per-pixel quality gate — pixels below this get NaN tau)")
print(f"Colour range        : {TAU_COLOUR_MIN}–{TAU_COLOUR_MAX} ns")

print("\nImports & IRF OK ✓")


Machine IRF       : machine_irf_default.npy  |  526 bins  |  peak bin 29
FIT_BG=True  FIT_SIGMA=False  HAS_TAIL=False
n_exp=3  tau=[0.145, 45.0] ns

INTENSITY_THRESHOLD : 3  (background mask — applied to both passes)
MIN_PHOTONS (NNLS)  : 10  (per-pixel quality gate — pixels below this get NaN tau)
Colour range        : 0.0–5.0 ns

Imports & IRF OK ✓


In [3]:
# Discover XLIF files and infer PTU basenames
xlif_files = sorted(XLIF_FOLDER.glob("*.xlif"))
print(f"Found {len(xlif_files)} XLIF file(s):")
for x in xlif_files:
    print(f"  {x.name}")

if not xlif_files:
    print("\nNo XLIF files found. Checking subdirectories...")
    xlif_files = sorted(XLIF_FOLDER.rglob("*.xlif"))
    if xlif_files:
        print(f"Found {len(xlif_files)} XLIF file(s) in subdirectories:")
        for x in xlif_files:
            print(f"  {x.relative_to(XLIF_FOLDER)}")

Found 75 XLIF file(s):
  C1_R1.xlif
  C1_R10.xlif
  C1_R11.xlif
  C1_R12.xlif
  C1_R13.xlif
  C1_R14.xlif
  C1_R2.xlif
  C1_R3.xlif
  C1_R4.xlif
  C1_R5.xlif
  C1_R6.xlif
  C1_R7.xlif
  C1_R8.xlif
  C1_R9.xlif
  C2_R1.xlif
  C2_R10.xlif
  C2_R11.xlif
  C2_R12.xlif
  C2_R13.xlif
  C2_R2.xlif
  C2_R3.xlif
  C2_R4.xlif
  C2_R5.xlif
  C2_R6.xlif
  C2_R8, C2_R7.xlif
  C2_R9.xlif
  C3_R1.xlif
  C3_R10.xlif
  C3_R11.xlif
  C3_R12.xlif
  C3_R13.xlif
  C3_R2.xlif
  C3_R3.xlif
  C3_R4.xlif
  C3_R5.xlif
  C3_R6.xlif
  C3_R7.xlif
  C3_R8.xlif
  C3_R9.xlif
  C4_R1.xlif
  C4_R10.xlif
  C4_R11.xlif
  C4_R12.xlif
  C4_R13.xlif
  C4_R2.xlif
  C4_R3.xlif
  C4_R4.xlif
  C4_R5.xlif
  C4_R6.xlif
  C4_R7.xlif
  C4_R8.xlif
  C4_R9.xlif
  C5_R1.xlif
  C5_R10.xlif
  C5_R11.xlif
  C5_R12.xlif
  C5_R13.xlif
  C5_R2.xlif
  C5_R3.xlif
  C5_R4.xlif
  C5_R5.xlif
  C5_R6.xlif
  C5_R7.xlif
  C5_R8.xlif
  C5_R9.xlif
  C6_R1.xlif
  C6_R2.xlif
  C6_R3.xlif
  C6_R4.xlif
  C6_R5.xlif
  C6_R6.xlif
  C6_R7.xlif
  C6_R8.xlif


In [ ]:
from flimkit.PTU.stitch        import fit_flim_tiles
from flimkit.FLIM.assemble     import derive_global_tau, save_assembled_maps
from flimkit.utils.lifetime_image import make_lifetime_image, make_component_rgb_tiff
import argparse, gc
from tqdm.notebook import tqdm


def _assemble_fixed(tile_results, canvas_height, canvas_width, n_exp):
    """
    Intensity-weighted assembly with per-key fitted-pixel denominator.

    Bug fixed vs assemble_tile_maps: the old version used a single weight
    accumulator (total photons) as the denominator for every key, so pixels
    below MIN_PHOTONS (unfitted, NaN tau) produced 0/photons = 0.0 instead
    of NaN. This made them appear as tau=0 in lifetime maps and images.

    Fix: maintain wt_fitted[key] — Σ intensity only where that key is finite
    (pixel was actually fitted). Unfitted pixels stay NaN throughout.
    """
    H, W = canvas_height, canvas_width
    keys_all = (['tau_mean_amp', 'chi2'] +
                [f'tau{k}' for k in range(1, n_exp + 1)] +
                [f'a{k}'   for k in range(1, n_exp + 1)])

    wsum      = {k: np.zeros((H, W), dtype=np.float64) for k in keys_all}
    wt_fitted = {k: np.zeros((H, W), dtype=np.float64) for k in keys_all}
    int_sum   = np.zeros((H, W), dtype=np.float32)
    coverage  = np.zeros((H, W), dtype=np.uint16)

    for tr in tile_results:
        pm = tr.get('pixel_maps')
        if pm is None:
            continue
        y0, x0 = tr['pixel_y'], tr['pixel_x']
        th, tw  = tr['tile_h'],  tr['tile_w']
        y1, x1  = min(y0 + th, H), min(x0 + tw, W)
        dy, dx  = y1 - y0, x1 - x0
        tile_int = np.asarray(
            pm.get('intensity', np.zeros((th, tw))), dtype=np.float64)[:dy, :dx]
        int_sum[y0:y1, x0:x1]  += tile_int.astype(np.float32)
        coverage[y0:y1, x0:x1] += 1
        for key in keys_all:
            src = pm.get(key)
            if src is None:
                continue
            src    = np.asarray(src, dtype=np.float64)[:dy, :dx]
            fitted = np.isfinite(src)
            wsum[key][y0:y1, x0:x1]      += np.where(fitted, src * tile_int, 0.0)
            wt_fitted[key][y0:y1, x0:x1] += np.where(fitted, tile_int,       0.0)

    canvas = {
        'intensity': (int_sum / np.where(coverage > 0, coverage, 1)).astype(np.float32),
        'coverage':  coverage.astype(np.float32),
    }
    for key in keys_all:
        safe_wt = np.where(wt_fitted[key] > 0, wt_fitted[key], np.nan)
        canvas[key] = (wsum[key] / safe_wt).astype(np.float32)
    return canvas


# Prepare CSV for streaming results
csv_path = OUTPUT_BASE / "batch_roi_fit_global_summary.csv"
csv_header_written = False

for xlif_idx, xlif_path in enumerate(xlif_files):
    xlif_path    = Path(xlif_path)
    ptu_basename = xlif_path.stem
    roi_clean    = ptu_basename.replace(' ', '_')
    output_dir   = OUTPUT_BASE / roi_clean
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  ROI {xlif_idx + 1}/{len(xlif_files)}: {ptu_basename}")
    print(f"{'='*60}")

    canvas         = None
    tile_results   = None
    global_summary = None
    fit_args       = None
    result         = None
    df_row         = None

    try:
        fit_args = argparse.Namespace(
            nexp            = N_EXP,
            tau_min         = TAU_MIN_NS,
            tau_max         = TAU_MAX_NS,
            optimizer       = 'de',
            restarts        = 1,
            de_population   = DE_POPSIZE,
            de_maxiter      = DE_MAXITER,
            workers         = WORKERS,
            binning         = 1,
            min_photons          = _MIN_PHOTONS,
            intensity_threshold  = _INT_THRESHOLD,
            machine_irf     = str(MACHINE_IRF_PATH) if MACHINE_IRF_PATH else str(_irf_path),
            tau_display_min = TAU_COLOUR_MIN,
            tau_display_max = TAU_COLOUR_MAX,
            irf_xlsx_dir    = None,
            irf_xlsx_map    = None,
            ptu_basename    = ptu_basename,
            xlif            = str(xlif_path),
            ptu_dir         = str(PTU_FOLDER),
            output_dir      = str(output_dir),
        )

        print(f"  [1/5] Fitting tiles...")
        tile_results, canvas_height, canvas_width = fit_flim_tiles(
            xlif_path    = xlif_path,
            ptu_dir      = PTU_FOLDER,
            output_dir   = output_dir,
            args         = fit_args,
            ptu_basename = ptu_basename,
            rotate_tiles = ROTATE_TILES,
            verbose      = False,
        )

        if not tile_results:
            print(f"  WARNING: No tiles fitted")
            result = {'roi': ptu_basename, 'status': 'No tiles fitted'}
            df_row = pd.DataFrame([result])
            df_row.to_csv(csv_path, index=False,
                          mode='a', header=not csv_header_written)
            csv_header_written = True
            continue

        print(f"  [2/5] Assembling canvas ({len(tile_results)} tiles)...")
        canvas = _assemble_fixed(tile_results, canvas_height, canvas_width, N_EXP)
        del tile_results
        tile_results = None
        gc.collect()

        print(f"  [3/5] Computing global summary...")
        global_summary = derive_global_tau(canvas, n_exp=N_EXP)
        tau_mean = global_summary.get('tau_mean_amp_global_ns', float('nan'))
        n_pixels = global_summary.get('n_pixels_fitted', 0)
        print(f"        tau_mean = {tau_mean:.4f} ns  |  {n_pixels:,} pixels")

        print(f"  [4/5] Saving maps & images...")
        save_assembled_maps(
            canvas               = canvas,
            global_summary       = global_summary,
            output_dir           = output_dir,
            roi_name             = roi_clean,
            n_exp                = N_EXP,
            tau_display_min      = TAU_COLOUR_MIN,
            tau_display_max      = TAU_COLOUR_MAX,
            intensity_display_min= 0.0,
            intensity_display_max= INTENSITY_DISPLAY_MAX,
        )

        make_lifetime_image(
            canvas          = canvas,
            output_dir      = output_dir,
            roi_name        = roi_clean,
            tau_min_ns      = TAU_COLOUR_MIN,
            tau_max_ns      = TAU_COLOUR_MAX,
            smooth_sigma_px = 2.0,
            gamma           = 0.4,
            verbose         = False,
        )

        make_component_rgb_tiff(
            canvas     = canvas,
            output_dir = output_dir,
            roi_name   = roi_clean,
            n_exp      = N_EXP,
            verbose    = False,
        )

        print(f"  [5/5] Writing CSV row...")
        result = {'roi': ptu_basename, 'status': 'OK', **global_summary}
        df_row = pd.DataFrame([result])
        df_row.to_csv(csv_path, index=False,
                      mode='a', header=not csv_header_written)
        csv_header_written = True
        print(f"  OK: {ptu_basename}")

    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"  ERROR: {ptu_basename}: {type(e).__name__}: {str(e)[:120]}")
        result = {'roi': ptu_basename, 'status': f'ERROR: {type(e).__name__}: {str(e)[:80]}'}
        df_row = pd.DataFrame([result])
        df_row.to_csv(csv_path, index=False,
                      mode='a', header=not csv_header_written)
        csv_header_written = True

    finally:
        # Only explicit del statements reliably free memory in Python loops.
        # del locals()[var] and exec('del x') do not work in Python.
        del canvas, tile_results, global_summary, fit_args, result, df_row
        gc.collect()

print(f"\n{'='*60}")
print(f"All ROIs processed")
print(f"CSV: {csv_path}")
print(f"{'='*60}")



  ROI 1/75: C1_R1
  [1/5] Fitting tiles...
  Next-period artefact at bin 492 (47.71 ns). Truncating fit window.
  Cost function: poisson
  bg initial guess = 2625.500 cts/bin, upper bound = 5251.000 (free param)
  σ broadening: fixed at 0
  Fit window: bins 1–464 (0.10–44.99 ns), 463 bins
  Differential evolution: popsize=15, maxiter=1000, workers=-1
  Running final LM polish...
  Per-pixel fitting: 512×512=262144 pixels (τ fixed, amplitudes + bg free) …


  Per-pixel rows: 100%|██████████| 512/512 [00:00<00:00, 1140.30it/s]


  Fitted: 5689/262144  |  Skipped (<10 ph): 256455  |  0.4s
  Per-pixel fitting: 512×512=262144 pixels (τ fixed, amplitudes + bg free) …


  Per-pixel rows: 100%|██████████| 512/512 [00:01<00:00, 319.92it/s] 


  Fitted: 40260/262144  |  Skipped (<10 ph): 221884  |  1.6s
  Per-pixel fitting: 512×512=262144 pixels (τ fixed, amplitudes + bg free) …


...

  Per-pixel rows: 100%|██████████| 512/512 [00:04<00:00, 110.40it/s]


  Fitted: 127771/262144  |  Skipped (<10 ph): 134373  |  4.6s
  Per-pixel fitting: 512×512=262144 pixels (τ fixed, amplitudes + bg free) …


  Per-pixel rows: 100%|██████████| 512/512 [00:04<00:00, 106.34it/s]


  Fitted: 132838/262144  |  Skipped (<10 ph): 129306  |  4.8s
  Per-pixel fitting: 512×512=262144 pixels (τ fixed, amplitudes + bg free) …


  Per-pixel rows:  34%|███▎      | 172/512 [00:01<00:02, 125.69it/s]

In [ ]:
# Load and display the final summary CSV
csv_path = OUTPUT_BASE / "batch_roi_fit_global_summary.csv"
df_results = pd.read_csv(csv_path)

print(f"\nGlobal summary CSV → {csv_path}")
print(f"Shape: {df_results.shape[0]} ROIs × {df_results.shape[1]} columns\n")
print(df_results.to_string())

In [ ]:
import matplotlib.pyplot as plt

# Summary statistics
ok = df_results[df_results.status == "OK"].copy() if 'status' in df_results.columns else df_results.copy()

if len(ok) > 0:
    print("\n── Global Lifetime Summary Statistics ───────────────────────")
    stat_cols = [
        'tau_mean_amp_global_ns', 'tau_std_amp_global_ns',
        'tau_median_amp_global_ns',
        'n_pixels_fitted'
    ]
    stat_cols = [c for c in stat_cols if c in ok.columns]
    if stat_cols:
        print(ok[stat_cols].describe().round(4).to_string())
    
    # Plot tau distributions if multiple ROIs
    if len(ok) > 1 and 'tau_mean_amp_global_ns' in ok.columns:
        fig, axes = plt.subplots(1, min(3, len(ok)+1), figsize=(4*min(3, len(ok)+1), 3.5))
        if not isinstance(axes, np.ndarray):
            axes = [axes]
        
        if 'tau_mean_amp_global_ns' in ok.columns:
            axes[0].bar(ok['roi'], ok['tau_mean_amp_global_ns'],
                        color='#028090', edgecolor='white', alpha=0.85, width=0.6)
            axes[0].set_ylabel('τ mean (ns)')
            axes[0].set_title('τ mean (amplitude-weighted)')
            axes[0].grid(True, alpha=0.3, axis='y')
            axes[0].tick_params(axis='x', rotation=45)
        
        if len(axes) > 1 and 'tau_std_amp_global_ns' in ok.columns:
            axes[1].bar(ok['roi'], ok['tau_std_amp_global_ns'],
                        color='#e74c3c', edgecolor='white', alpha=0.85, width=0.6)
            axes[1].set_ylabel('τ std (ns)')
            axes[1].set_title('τ std (amplitude-weighted)')
            axes[1].grid(True, alpha=0.3, axis='y')
            axes[1].tick_params(axis='x', rotation=45)
        
        if len(axes) > 2 and 'n_pixels_fitted' in ok.columns:
            axes[2].bar(ok['roi'], ok['n_pixels_fitted'],
                        color='#2ecc71', edgecolor='white', alpha=0.85, width=0.6)
            axes[2].set_ylabel('# fitted pixels')
            axes[2].set_title('Fitted pixel count')
            axes[2].grid(True, alpha=0.3, axis='y')
            axes[2].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.savefig(OUTPUT_BASE / "batch_roi_fit_summary_plots.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"\nPlot saved → {OUTPUT_BASE / 'batch_roi_fit_summary_plots.png'}")
else:
    print("No successful ROI fits to summarize.")